# Supply Chain Twin: Google Colab notebook (data on Google Drive)

Generated by `scripts/build_notebook.py` (at commit `d209912`). The setup cell checks out the
code from GitHub (`REF`, default `main`) into `/content/SupplyTwinAI` and installs it, so Colab
runs exactly the pushed code; the checked-out commit is printed. To change the code, push to
GitHub (or set `REF` to another branch, tag or commit) and re-run the setup cell.

**Google Drive layout** (created on first run; change `DRIVE` in the setup cell to move it):

```
MyDrive/SupplyTwinAI/
  data/raw/provenance.json          dataset (pinned revision), about 110 MB
  data/raw/data/train.parquet
  data/raw/data/eval.parquet        locked final test: integrity-checked, never loaded
  runs/                             all run outputs (metrics, predictions, posteriors)
```

The dataset is read from Drive and copied to the Colab disk for fast reads. If it is not on
Drive yet, the pinned revision is downloaded once and saved there. You can also upload a
local `data/raw/` folder (with `provenance.json`) to that place yourself.

**How to use:** run *Setup*, *Helpers* and *Data* (Runtime → Run all is fine), then the steps
you need. Finished steps keep their outputs on Drive, so you can skip them later.
`FAST = True` (in the helpers cell) keeps every step to a few minutes; set `FAST = False` for
the full runs. If Colab restarts the session, re-run *Setup* and *Helpers* first.
Use a CPU runtime with many cores: the installed JAX is CPU-only, so a GPU is not used.

Rules that every step follows are in `CLAUDE.md` of the repository: rolling-origin
backtesting without leakage, point-in-time features, the official `eval` split is never read.
Sales are normalized units; reconstructed demand is never presented as observed.

## Setup

In [1]:
import os, sys, pathlib, shutil, subprocess
from google.colab import drive

REPO = "https://github.com/moghbaie/SupplyTwinAI.git"
REF = "main"  # branch, tag or commit to run
DRIVE = pathlib.Path("/content/drive/MyDrive/SupplyTwinAI")  # dataset and run outputs live here
ROOT = pathlib.Path("/content/SupplyTwinAI")  # code checkout (Colab disk, wiped per runtime)

def sh(cmd):
    """Run a shell command in ROOT; stop the cell if it fails."""
    subprocess.run(cmd, shell=True, check=True, cwd=ROOT)

drive.mount("/content/drive")
for d in (DRIVE / "data" / "raw", DRIVE / "runs", ROOT):
    d.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

# Check out the code (works in an empty folder and over leftovers of an older notebook;
# untracked files such as data/ and the runs link are kept, tracked files are overwritten).
if not (ROOT / ".git").exists():
    sh(f"git init -q && git remote add origin {REPO}")
sh(f"git fetch -q --depth 1 origin {REF} && git checkout -q --force FETCH_HEAD")
sh('git log -1 --format="code: commit %h (%cd) %s" --date=short')

if not os.path.islink("runs"):  # run outputs go straight to Drive
    if os.path.isdir("runs"):
        os.rmdir("runs")  # only succeeds if empty, so nothing is lost
    os.symlink(DRIVE / "runs", "runs")

# Install with all extras (no --force-reinstall: it would upgrade Colab's numpy and force a
# session restart). The import check runs in a fresh process, like the run cells below.
sh('pip install -q -e ".[dev,models,bayes]"')
sh('python -c "import sctwin.censoring, sctwin.demand_model; print(\'package import OK\')"')
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))  # make the package importable in this kernel too
print("code:", ROOT, "| data and runs:", DRIVE, "| CPU cores:", os.cpu_count())

Mounted at /content/drive
code: /content/SupplyTwinAI | data and runs: /content/drive/MyDrive/SupplyTwinAI | CPU cores: 12


## Helpers

In [2]:
import copy, glob, json
import pandas as pd
import yaml

os.chdir(ROOT)

FAST = False          # small, quick runs; set False for the full runs
CORES = os.cpu_count() or 2

def config_with(path, updates, name):
    """Copy a YAML config with nested updates applied; returns the new path."""
    cfg = yaml.safe_load(open(path, encoding="utf-8"))
    def merge(a, b):
        for k, v in b.items():
            if isinstance(v, dict) and isinstance(a.get(k), dict):
                merge(a[k], v)
            else:
                a[k] = v
    merge(cfg, copy.deepcopy(updates))
    out = f"configs/nb_{name}.yaml"
    yaml.safe_dump(cfg, open(out, "w", encoding="utf-8"), sort_keys=False)
    return out

def latest(pattern):
    files = sorted(glob.glob(pattern, recursive=True), key=os.path.getmtime)
    if not files:
        raise FileNotFoundError(pattern)
    print("reading", files[-1])
    return json.load(open(files[-1], encoding="utf-8"))

def model_table(models):
    rows = {}
    for m, r in models.items():
        iv = r.get("intervals", {})
        rows[m] = {"mae": r.get("mae", r.get("mae_median")), "bias": r.get("bias"),
                   "wis": r.get("wis"), "cov50": iv.get("0.5", {}).get("coverage"),
                   "cov90": iv.get("0.9", {}).get("coverage")}
    return pd.DataFrame(rows).T.round(4)

## Run: tests (synthetic, no data needed)
The leakage tests run for both units, "day" and "hour". Warnings are errors, except
deprecation warnings: Colab's preinstalled libraries raise them inside each other (e.g.
matplotlib calling deprecated pyparsing), which is not this project's code. Local runs and CI
keep plain `-W error`.

In [3]:
!python -m pytest -W error -W "default::DeprecationWarning" -m "not data" -q -p no:cacheprovider

........................................................................ [ 64%]
........................................                                 [100%]
=============================== warnings summary ===============================
tests/test_baseline.py::test_runner_output_and_protocol[day]
  /usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
    prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))

tests/test_baseline.py::test_runner_output_and_protocol[day]
  /usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
    parse = parser.parseString(pattern)

tests/test_baseline.py::test_runner_output_and_protocol[day]
  /usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cach

## Data from Google Drive (run before any step)
Uses the dataset on Drive if present (checksums and pinned revision verified); otherwise
downloads the pinned revision once and saves it to Drive. Copies it to the Colab disk.

In [4]:
import json, yaml
from sctwin.data import verify_checksums

os.chdir(ROOT)
raw_drive = DRIVE / "data" / "raw"
pinned = yaml.safe_load(open("configs/small.yaml"))["dataset"]["revision"]
skip_cache = shutil.ignore_patterns(".cache")
if (raw_drive / "provenance.json").exists():
    print("dataset found on Drive:", raw_drive)
    shutil.copytree(raw_drive, "data/raw", dirs_exist_ok=True, ignore=skip_cache)
else:
    print("dataset not on Drive yet: downloading the pinned revision once")
    !python scripts/fetch_data.py
    shutil.copytree("data/raw", raw_drive, dirs_exist_ok=True, ignore=skip_cache)
    print("saved to Drive:", raw_drive)
prov = json.load(open("data/raw/provenance.json"))
assert prov["revision"] == pinned, f"Drive has revision {prov['revision']}, config pins {pinned}"
checks = verify_checksums("data/raw")
assert all(checks.values()), f"checksum mismatch: {checks}"
print("revision", pinned[:12], "checksums OK:", checks)

dataset found on Drive: /content/drive/MyDrive/SupplyTwinAI/data/raw
revision c374e85975c8 checksums OK: {'train': True, 'eval': True}


## Run: Step 1, audit and real-data tests
Re-runs the audit (it re-writes the identical split manifest) and the real-data tests,
including the real-data leakage checks.

In [5]:
!python scripts/audit_data.py
!python -m pytest -m data -q -p no:cacheprovider

{
  "gates": {
    "train.schema_ok": true,
    "train.no_duplicate_keys": true,
    "train.dates_parse": true,
    "train.arrays_len_24": true,
    "train.no_negative_or_nan_sales": true,
    "train.status_binary": true,
    "eval.schema_ok": true,
    "eval.no_duplicate_keys": true,
    "eval.dates_parse": true,
    "eval.arrays_len_24": true,
    "eval.no_negative_or_nan_sales": true,
    "eval.status_binary": true,
    "train.hierarchy_strict_tree": true,
    "cross.no_overlapping_keys": true,
    "cross.test_strictly_after_train": true
  },
  "all_gates_pass": true,
  "series_counts": {
    "total": 50000,
    "in_sample": 43883,
    "holdout_store_only": 4758,
    "holdout_product_only": 1242,
    "holdout_both": 117
  },
  "runtime_s": 48.7
}
......                                                                   [100%]


## Run: Step 3, baseline backtest (observed sales)
Rolling origin; train on feature times t <= T - L - H; predict (T + L, T + L + H].
Full day run: about 15 min on 12 cores and about 6 GB RAM. FAST uses 200 series and 2 folds.

In [6]:
cfg3 = config_with("configs/baselines.yaml", {"lgbm": {"num_threads": CORES}}, "baselines")
smoke = "--smoke" if FAST else ""
!python scripts/run_baselines.py --unit day --config {cfg3} {smoke}
!python scripts/run_baselines.py --unit hour --config {cfg3} {smoke}

[day] origin 74: {'origin': 74, 'train_rows': 3276000, 'test_rows': 307181, 'train_t_range': [28, 66], 'glm_phi': 0.2849907472692629, 'lgbm_q_crossing_frac': 0.02529453319052936, 'seconds': 413.8}
[day] origin 81: {'origin': 81, 'train_rows': 3528000, 'test_rows': 307181, 'train_t_range': [32, 73], 'glm_phi': 0.2729138851113119, 'lgbm_q_crossing_frac': 0.02136850912003021, 'seconds': 428.9}

== day: 2 folds, 43883 series, validation window
  clim     mae=0.4775 wis=0.3059 cov90=0.670
  glm      mae=0.4184 wis=0.2508 cov90=0.888
  lgbm     mae=0.3919 wis=nan cov90=nan
  lgbm_q   mae=0.3903 wis=0.2300 cov90=0.856
  snaive   mae=0.4842 wis=nan cov90=nan

written to runs/step3/day/1cdd2e3f9599  (857.9 s)
[hour] origin 1822: {'origin': 1822, 'train_rows': 1008000, 'test_rows': 12000, 'train_t_range': [1299, 1797], 'glm_phi': 1.302975834062331, 'lgbm_q_crossing_frac': 0.0365, 'seconds': 136.9}
[hour] origin 1846: {'origin': 1846, 'train_rows': 1008000, 'test_rows': 12000, 'train_t_range': [1

In [7]:
for unit in ("day", "hour"):
    m = latest(f"runs/step3*/{unit}/*/metrics.json")
    print(unit, "origins", m["protocol"]["origins"][:5], "...")
    display(model_table(m["metrics"]["overall"]["models"]))
    display(pd.DataFrame({o: {k: v.get("wis", v.get("mae")) for k, v in f["models"].items()}
                          for o, f in m["metrics"]["folds"].items()}).T.round(4))

reading runs/step3/day/1cdd2e3f9599/metrics.json
day origins [74, 81] ...


,mae,bias,wis,cov50,cov90
clim,0.4775,NaN,0.3059,0.3938,0.6705
glm,0.4184,-0.0499,0.2508,0.5365,0.8883
lgbm,0.3919,-0.0871,NaN,NaN,NaN
lgbm_q,0.3903,NaN,0.2300,0.4514,0.8563
snaive,0.4842,-0.0841,NaN,NaN,NaN


,clim,glm,lgbm,lgbm_q,snaive
74,0.2776,0.2357,0.3599,0.2075,0.4653
81,0.3343,0.2659,0.4240,0.2526,0.5031


reading runs/step3/hour/4c76dcfb809e/metrics.json
hour origins [1822, 1846, 1870, 1894, 1918] ...


,mae,bias,wis,cov50,cov90
clim,0.0404,NaN,0.0258,0.8619,0.9452
glm,0.0520,0.0009,0.0278,0.7989,0.9672
lgbm,0.0477,-0.0041,NaN,NaN,NaN
lgbm_q,0.0400,NaN,0.0249,0.8050,0.9484
snaive,0.0561,-0.0005,NaN,NaN,NaN


,clim,glm,lgbm,lgbm_q,snaive
1822,0.0215,0.0235,0.0415,0.0210,0.0481
1846,0.0215,0.0228,0.0414,0.0207,0.0476
1870,0.0231,0.0251,0.0433,0.0225,0.0472
1894,0.0296,0.0314,0.0521,0.0283,0.0606
1918,0.0299,0.0314,0.0525,0.0280,0.0618
1942,0.0230,0.0254,0.0446,0.0227,0.0603
1966,0.0232,0.0255,0.0441,0.0228,0.0512
1990,0.0249,0.0273,0.0461,0.0246,0.0546
2014,0.0241,0.0263,0.0459,0.0237,0.0539
2038,0.0253,0.0278,0.0473,0.0249,0.0545


## Run: Step 4, censoring benchmark (proxy truth)
Demand-driven stockouts injected into fully in-stock days; calibration on the fit window only.
Full run: about 1.5 min for 5,000 series. FAST uses 300 series.

In [8]:
smoke = "--smoke" if FAST else ""
!python scripts/run_censoring_benchmark.py {smoke}

calibrated: {'coverage': 1.15, 'coverage_sd': 0.6, 'leak_first': 0.16711784156890871, 'leak_later': 0.07448787932732963, 'p_restock': 0.187, 'restock_mean_hours': 5.75}

real (fit)  : {'n_days': 226008.0, 'stockout_rate': 0.38104845846164737, 'restock_rate': 0.11002090106827682, 'restock_mean_hours': 3.1325593667546174, 'first_hour_q25': 12.0, 'first_hour_q50': 16.0, 'first_hour_q75': 18.0}
injected    : {'n_days': 39409.0, 'stockout_rate': 0.40891674490598595, 'restock_rate': 0.1132485262178095, 'restock_mean_hours': 3.30027397260274, 'first_hour_q25': 12.0, 'first_hour_q50': 16.0, 'first_hour_q75': 18.0}
positions real {'-3': 1.5624148929943753, '-2': 1.6197055401500773, '-1': 2.3881983542684426, '+0': 0.1979479901406548, '+1': 0.08552556177435763}
          inj  {'-3': 1.3659604140612622, '-2': 1.469277422895018, '-1': 1.775485265532637, '+0': 0.22752854263548747, '+1': 0.09620379510147717}

== 16628 stockout days / 39409 days, 119351 censored hours
  hourly hour_mean    mae=0.0718 

In [9]:
m = latest("runs/step4*/*/metrics.json")
print("calibrated injection:", m["calibrated_config"])
display(pd.DataFrame({"real": m["real_fit_window"]["stats"],
                      "injected": m["injected"]["stats"]}).round(3))
display(pd.DataFrame({"real": m["real_fit_window"]["position_ratios"],
                      "injected": m["injected"]["position_ratios"]}).round(3))
print("daily totals on stockout days")
display(model_table(m["benchmark"]["daily"]["overall"]["models"]))

reading runs/step4/8ba303c79bd4/metrics.json
calibrated injection: {'coverage': 1.15, 'coverage_sd': 0.6, 'leak_first': 0.16711784156890871, 'leak_later': 0.07448787932732963, 'p_restock': 0.187, 'restock_mean_hours': 5.75}


,real,injected
n_days,226008.000,39409.000
stockout_rate,0.381,0.409
restock_rate,0.110,0.113
restock_mean_hours,3.133,3.300
first_hour_q25,12.000,12.000
first_hour_q50,16.000,16.000
first_hour_q75,18.000,18.000


,real,injected
-3,1.562,1.366
-2,1.620,1.469
-1,2.388,1.775
+0,0.198,0.228
+1,0.086,0.096


daily totals on stockout days


,mae,bias,wis,cov50,cov90
hour_mean,0.2265,-0.1429,NaN,NaN,NaN
observed,0.4854,-0.4854,NaN,NaN,NaN
profile,0.3304,0.0374,NaN,NaN,NaN
profile_pre,0.3450,0.0394,NaN,NaN,NaN


## Run: Step 5, Bayesian model ladder (M0, M1, M2, M1d, M2d)
NumPyro NUTS. FAST: 4 series and short chains (a few minutes). Full: 20 series, 4 chains x
(1000 + 1000) draws; chains run in parallel, one per CPU core (up to 4), so it takes hours on a
2-core runtime. `--part real` and `--part recovery` split the run in two.
M3 = M2 with `--n-series 100 --models M2`. M1d/M2d are exploratory (added after the plan).

In [ ]:
chains = min(4, CORES)
cfg5 = config_with("configs/bayes.yaml", {"mcmc": {"num_chains": chains}}, "bayes")
smoke = "--smoke" if FAST else ""
!python scripts/run_bayes.py --config {cfg5} --part all {smoke}

fit: 20 series x 76 days; benchmark: 174 days, 71 with a stockout
[M0] fitting ...
[M0] 656.1 s, divergences=0, max R-hat=1.019, min ESS=253
/usr/local/lib/python3.13/dist-packages/arviz_stats/loo/loo_helper.py:1169: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.67 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(
[M1] fitting ...
[M1] 597.0 s, divergences=0, max R-hat=1.008, min ESS=319
[M2] fitting ...
[M2] 2745.7 s, divergences=0, max R-hat=1.011, min ESS=237
[M1d] fitting ...
[M1d] 598.7 s, divergences=0, max R-hat=1.022, min ESS=294
[M2d] fitting ...
[M2d] 2888.6 s, divergences=0, max R-hat=1.024, min ESS=275
/usr/local/lib/python3.13/dist-packages/arviz_stats/loo/loo_helper.py:1169: UserWarnin

In [ ]:
m = latest("runs/step5*/*/metrics_all.json")
real = m["real"]
keys = ("runtime_s", "divergences", "max_rhat", "min_ess")
display(pd.DataFrame({k: {x: v["diagnostics"][x] for x in keys}
                      for k, v in real["models"].items()}).T)
print("daily totals on stockout days (proxy truth); baselines are point estimates")
display(model_table(real["daily"]["overall"]["models"]))
print("PSIS-LOO on in-stock hours:", real["loo_compare"])
for name, r in m["recovery"]["models"].items():
    print("recovery", name, r["mean_window_demand"])
    display(pd.DataFrame(r["scalars"]).T)

## Run: Step 5, simulation-based calibration of M1
Ranks of the true parameters among posterior draws should be uniform (chi-square p-values not
small after Holm). Full: 200 replications (about 40 min). FAST: 20.

In [ ]:
reps = 20 if FAST else 200
!python scripts/run_sbc.py --reps {reps}
s = latest("runs/sbc/*/ranks.json")
display(pd.DataFrame(s["scalars"]).T)
print("min Holm-adjusted p:", s["min_p_holm"])
print("divergent replications:", s["divergent_reps"])